# Level 2 · Limonene SOA: explicit chemistry, volatility and partitioning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/notebooks/02_limonene_soa.ipynb)

In this notebook 10 ppb of limonene is oxidized by O₃, OH and NO₃ in a box under constant,
chamber-like light. The gas-phase chemistry is explicit: the Master Chemical Mechanism (MCM v3.3.1)
limonene subset, with about 725 species and 2135 reactions, solved as a stiff system of ODEs. Each
closed-shell organic product gets a saturation concentration c\*. The products are lumped into
half-decade volatility classes, and the classes partition between the gas and a seed aerosol. That
is the Level 1 physics, now fed by real chemistry. The Limonene SOA tab of the web app runs the same
model and preset with sliders. Here you can read the mechanism, change any input and write your own
analysis.

> **Read this first: the c\* values are placeholders.** Until a curated per-species table arrives,
> c\* comes from a formula that uses only the elemental composition (Li et al., 2016). With it,
> about 99% of the SOA is nitrogen-containing (organic nitrates and PANs), and with NOx = 0 there
> is almost no SOA. Treat this as a lesson in how much SOA predictions depend on c\*, not as a
> statement about limonene. The full list of model assumptions is in
> [ASSUMPTIONS.md](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/ASSUMPTIONS.md); A8 (c\*) and A10 (constant light) matter most here.

## Setup

Run the next two cells once per session.

- **Colab:** the first cell installs the box model and the tutorial helpers (1–2 minutes). Red pip
  "dependency conflicts" lines are expected. If a cell asks you to restart, click
  *Runtime › Restart session*, then *Runtime › Run all*.
- **Your own computer:** install the repository first ([README](https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/README.md#run-on-your-own-computer));
  the first cell then does nothing.
- `at.setup()` downloads the TUV-x photolysis data once (about 5 MB) and prints the versions.

**Run times.** The first model run in a session also compiles the model (JAX) and builds the
photolysis table. That adds up to about 30–40 s, less on a fast laptop. After that a 6-hour Level 2
run takes about 8–15 s on a laptop, and maybe twice that on Colab. The whole notebook takes about
2 minutes on a laptop and 3–5 minutes on Colab.

In [ ]:
# Colab: installs the tutorial helpers and the box model (first time: 1-2 minutes).
# Red pip "dependency conflicts" lines on Colab are expected and do not matter here.
import importlib.metadata
import sys

HELPERS_VERSION = "0.2.0"   # the aaar_tutorial version these notebooks need


def _older(version):
    return tuple(map(int, version.split("."))) < tuple(map(int, HELPERS_VERSION.split(".")))


try:
    installed = importlib.metadata.version("aaar-tutorial")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed is None or _older(installed):
    if "google.colab" not in sys.modules:
        raise ModuleNotFoundError(
            f"This notebook needs the tutorial helpers {HELPERS_VERSION}, found {installed or 'none'}. Pick the "
            "kernel of the repository's .venv, update it (git pull, then uv sync or pip install -e .), then "
            "restart the kernel.")
    %pip install -q "aaar-tutorial @ git+https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial@main"
    importlib.invalidate_caches()           # so the next cell finds the new packages
loaded = getattr(sys.modules.get("aaar_tutorial"), "__version__", None)
if loaded is not None and _older(loaded):
    raise RuntimeError("This session still has older tutorial helpers loaded. Colab: Runtime > Restart "
                       "session, then Runtime > Run all. Locally: Kernel > Restart Kernel.")

In [ ]:
import numpy as np
import plotly.graph_objects as go

import aaar_tutorial as at
from aaar_tutorial import config, plots

at.setup()
PRESET = "level2_limonene_soa"

## Learning goals

By the end of this notebook you should be able to:

1. Say what an explicit mechanism contains (species, reactions, rate constants) and look up a
   reaction in it.
2. Work out which oxidant (O₃, OH or NO₃) removes limonene in this box, and why.
3. Explain how a product gets a c\* and is lumped into a half-decade volatility class.
4. Use absorptive partitioning, $F = 1/(1 + c^*/C_{OA})$, to read the volatility plot and check the
   model against it.
5. Read the top SOA contributors and explain why nearly all of them contain nitrogen here.
6. Change NOx, O₃, limonene and the seed, compare the runs, and say which conclusions depend on the
   c\* placeholder.

## 1. The scenario

The preset is a 6-hour run at 298 K and 50% RH with:

- 10 ppb limonene, 40 ppb O₃, 0.5 ppb NO, 2 ppb NO₂ and 1 ppb SO₂, plus background CO, CH₄ and H₂;
- a seed of 2000 particles cm⁻³ at 80 nm, 30% non-volatile organic by mass (the rest sulfate);
- **constant light** (chamber-like): the TUV-x spectrum at a solar zenith angle of 30°, scaled so
  that J(NO₂) = 5×10⁻³ s⁻¹ (ASSUMPTIONS A10).

There is no dilution, deposition or wall loss. `at.describe` prints every input you can change,
grouped by section:

In [ ]:
at.describe(PRESET)

### Run the preset

`at.run` runs the model and shows a progress bar. The first run in a session also compiles the
model. If you are short on time, set `USE_PRECOMPUTED = True` to load a saved result of the
unchanged preset instead (instant). The experiments further down always run the model.

In [ ]:
USE_PRECOMPUTED = False   # Try this: True loads the saved result of the unchanged preset (no model run)

if USE_PRECOMPUTED:
    base = at.load_example(PRESET)
else:
    base = at.run(PRESET, name="preset")

for key, value in at.summary(base).items():
    print(f"{key:>15}  {value:.4g}")

## 2. Explicit gas-phase chemistry (MCM)

An explicit mechanism lists every reaction of every species. A reaction A + B → products runs at
$r = k\,[A][B]$, with $k$ in cm³ molec⁻¹ s⁻¹ (or $r = J[A]$ for photolysis, with $J$ in s⁻¹). Each
species then gets one ODE, production minus loss:

$$\frac{d[X_i]}{dt} = \sum_j \nu^{+}_{ij}\, r_j \;-\; \sum_j \nu^{-}_{ij}\, r_j ,
\qquad r_j = k_j\,[A_j]\,[B_j]$$

where $\nu^{+}_{ij}$ and $\nu^{-}_{ij}$ are how many molecules of species $i$ reaction $j$ makes
and uses. The lifetimes in this system range from fractions of a second (O¹D, OH) to months (CO), so
the equations are *stiff*. The model integrates them with an implicit, adaptive solver (diffrax
Kvaerno3) over 300-s coupling intervals. As in KPP (the Kinetic PreProcessor, a standard tool and
file format for chemical mechanisms), it holds every $k$ fixed within an interval (ASSUMPTIONS A12).

The mechanism file is the MCM v3.3.1 limonene subset in KPP format, and the model parses it
directly. `at.mechanism` parses it and evaluates every rate constant at the preset's temperature,
pressure and RH ([Notebook 4](04_inside_the_model.ipynb) shows how). Let's open it:

In [ ]:
chem = at.mechanism(PRESET)   # the mechanism, with rate constants k at the preset's T, p and RH
mech = chem.mechanism         # the .eqn file (725 species) plus the N2O5-uptake counter NA_HET (A1)

print(f"species:              {len(mech.species)}")
print(f"reactions:            {len(mech.reactions)}")
print(f"photolysis reactions: {sum(r.photolysis for r in mech.reactions)}")
print(f"peroxy radicals RO2:  {len(mech.ro2)}")

In [ ]:
chem.show_reactions("LIMONENE")   # Try this: "O3", "NO2", or any name from mech.species

Limonene has three sets of first steps: OH addition (three branches, scaled to the chamber-measured
k_OH = 1.47×10⁻¹⁰ cm³ molec⁻¹ s⁻¹; A4), ozonolysis (two Criegee intermediates) and NO₃ addition.
Each gives a peroxy radical (RO₂) or a Criegee intermediate. Hundreds of later reactions turn these
into closed-shell products: carbonyls, acids, hydroperoxides, nitrates and PANs.

**Follow one product.** C822PAN is the largest SOA contributor in the preset (Section 5). Where does
it come from, and where does it go?

In [ ]:
print("Made by:")
chem.show_reactions("C822PAN", role="product")
print("\nLost by:")
chem.show_reactions("C822PAN")   # Try this: "LIMALOOH" (a hydroperoxide) or "LIMANO3" (a hydroxy nitrate)

A PAN forms when an acyl peroxy radical (here C822CO3) meets NO₂, and it falls apart again
thermally. The decomposition rate KBPAN ≈ 4×10⁻⁴ s⁻¹ at 298 K means a gas-phase lifetime of about
40 minutes. PAN formation needs NO₂, so it ties the products to NOx. Keep both facts in mind for
Section 5.

### Oxidants and gases over time

Click legend entries to show or hide species. Look at how fast limonene is used up, what happens to
NO and NO₂, and the levels of OH, HO₂ and NO₃. Light is constant, so there is no day–night cycle.

In [ ]:
plots.gases(base)

In [ ]:
plots.radicals(base)

### Who oxidizes limonene?

The rate at which oxidant X removes limonene is $L_X = k_X\,[X]\,[\mathrm{LIM}]$. Integrating it
over the run gives each oxidant's share of the limonene removed:

$$\text{share}_X = \frac{\int k_X [X][\mathrm{LIM}]\,dt}{\sum_Y \int k_Y [Y][\mathrm{LIM}]\,dt}$$

The cell below takes $k_X$ from the mechanism and $[X]$ and $[\mathrm{LIM}]$ from the model
output, which is saved every 10 minutes. The integral is therefore approximate.

In [ ]:
result = base   # Try this: a run from Section 6, e.g. nox_runs[-1] or o3_runs[-1]

k_limonene = {}   # [cm3 molec-1 s-1], summed over the branches of each oxidant
for r in mech.reactions:
    if "LIMONENE" in r.reactants and len(r.reactants) == 2:
        oxidant = r.reactants[1 - r.reactants.index("LIMONENE")]
        k_limonene[oxidant] = k_limonene.get(oxidant, 0.0) + chem.k[r.label]

t_h = np.asarray(result.time_h)
ppb_to_cm3 = chem.air["M"] * 1e-9
oxidant_cm3 = {"O3": at.series(result, "O3") * ppb_to_cm3,
               "OH": at.series(result, "OH"),
               "NO3": at.series(result, "NO3")}
limonene_ppb = at.series(result, "LIMONENE")

fig = go.Figure()
removed_ppb = {}
for x, conc in oxidant_cm3.items():
    loss_ppb_per_h = k_limonene[x] * conc * limonene_ppb * 3600.0
    removed_ppb[x] = np.trapezoid(loss_ppb_per_h, t_h)
    fig.add_trace(go.Scatter(x=t_h, y=loss_ppb_per_h, name=f"{x}  (k = {k_limonene[x]:.2g})"))
fig.update_xaxes(title=config.TIME_AXIS_TITLE)
fig.update_yaxes(title="Loss rate (ppb h⁻¹)")

total = sum(removed_ppb.values())
print("Share of limonene removed:", ", ".join(f"{x} {100 * v / total:.0f}%" for x, v in removed_ppb.items()))
print(f"Sum of the integrals: {total:.2f} ppb; limonene used in the model: "
      f"{limonene_ppb[0] - limonene_ppb[-1]:.2f} ppb")
plots.style(fig, "Limonene loss rate by oxidant")

Per molecule, OH reacts with limonene about 700 000 times faster than O₃. But O₃ (40 ppb, or
10¹² molec cm⁻³) is more than a million times more abundant than OH (a few 10⁵ cm⁻³), so here O₃
removes about 60% of the limonene and OH most of the rest. NO₃ matters little under constant light
because it is photolysed within seconds. At night it would matter much more (Level 3). The two
totals printed above differ by about 10% because limonene falls fast early on, and 10-minute
outputs are too coarse to integrate that exactly.

## 3. From products to volatility classes

Only **closed-shell** organic products partition to the particles. Radicals (RO₂, RO, acyl peroxy)
react too fast to matter. The model finds the closed-shell species by valence parity: the number of
H + N atoms is even. Each one gets a c\* at 298 K from the Li et al. (2016) parameterization:

$$\log_{10} c^* = (n_C^0 - n_C)\,b_C \;-\; n_O\,b_O \;-\; 2\,\frac{n_C\,n_O}{n_C + n_O}\,b_{CO}
\;-\; n_N\,b_N \;-\; n_S\,b_S$$

$n_C$, $n_O$, $n_N$ and $n_S$ are atom counts, and the coefficients are fitted separately for each
composition class (CH, CHO, CHON, …; Li et al., 2016, Table 1). The products are lumped into 15
half-decade classes centred on log₁₀ c\* = −3, −2.5, …, 4, each ±0.25 decade wide. Products more
volatile than 10^4.25 µg m⁻³ stay in the gas phase, and anything below 10^−3.25 joins the lowest
class. For the temperature dependence each class gets ΔH_vap = 131 − 11 log₁₀ c\* kJ mol⁻¹
(Epstein et al., 2010).

The formula only counts atoms. It cannot tell a thermally labile PAN (–C(O)OONO₂) from a stable
nitrate (–ONO₂), and it does not see functional groups at all. A8 puts its typical uncertainty at
±1–2 decades.

In [ ]:
from tomas_jax.gaschem.volatility import assign_classes, is_closed_shell, log10_cstar_li2016

examples = {
    "LIMONENE": "the precursor",
    "LIMAL": "limonaldehyde, a first-generation product",
    "LIMONONIC": "limononic acid",
    "LIMALOOH": "a hydroperoxide (from limonaldehyde + OH)",
    "LIMANO3": "a hydroxy nitrate (LIMAO2 + NO)",
    "C822PAN": "a PAN, the top SOA contributor in the preset",
}   # Try this: add any species from mech.species
for name, what in examples.items():
    print(f"{name:10} {chem.formula(name):11} log10 c* = {log10_cstar_li2016(mech.atoms[name]):5.2f}   {what}")

classes = assign_classes(mech)   # the model's lumping (Li et al. c*, since no table is given)
print(f"\n{len(classes.species)} condensable species in {classes.class_cstar.size} classes; "
      f"c* source: {classes.source}")

The histogram shows the estimated c\* of every closed-shell organic in the mechanism. The grey band
is the range the partitioning classes cover, and the dashed line is the absorbing organic mass
C_OA at the end of the preset run. Species to the left of that line end up mostly in the particles.

In [ ]:
closed_shell = [s for s in mech.species if mech.atoms[s].get("C") and is_closed_shell(mech.atoms[s])]
log_cstar = {s: log10_cstar_li2016(mech.atoms[s]) for s in closed_shell}
has_n = {s: bool(mech.atoms[s].get("N")) for s in closed_shell}

fig = go.Figure()
for flag, label, color in ((False, "no N", config.PARTICLE_COLOR),
                           (True, "contains N (nitrates, PANs)", config.SERIES_COLORS[1])):
    fig.add_trace(go.Histogram(x=[log_cstar[s] for s in closed_shell if has_n[s] == flag], name=label,
                               xbins=dict(start=-6.25, end=12.25, size=0.5), marker_color=color,
                               opacity=0.7))
edge = 0.25   # half-width of a class [decades]
fig.add_vrect(x0=np.log10(classes.class_cstar.min()) - edge,
              x1=np.log10(classes.class_cstar.max()) + edge, fillcolor="rgba(0,0,0,0.07)",
              line_width=0, layer="below", annotation_text="partitioning classes",
              annotation_position="top left")
c_oa_end = at.series(base, "SOA_ugm3")[-1] + at.series(base, "POA_ugm3")[-1]
fig.add_vline(x=np.log10(c_oa_end), line=dict(color="#444444", dash="dash"),
              annotation_text=f"C_OA at 6 h ≈ {c_oa_end:.0f} µg m⁻³", annotation_position="top right")
fig.update_layout(barmode="overlay")
fig.update_xaxes(title="log₁₀ c* at 298 K (µg m⁻³)")
fig.update_yaxes(title="Number of species")
plots.style(fig, "Li et al. (2016) c* of the closed-shell organics in the mechanism")

## 4. Absorptive partitioning

The rule is the same as in Level 1, now applied to each class $j$. At equilibrium the particle
fraction is

$$F_j = \frac{C_{p,j}}{C_{p,j} + C_{g,j}} = \frac{1}{1 + c^*_j / C_{OA}},
\qquad C_{OA} = C_{nv} + \sum_j F_j\,\bigl(C_{p,j} + C_{g,j}\bigr)$$

$C_{OA}$ is the absorbing organic mass: the SOA plus the seed's non-volatile organic ($C_{nv}$, 0.77
µg m⁻³). Sulfate, nitrate and water do not absorb organics in this model. A class with
$c^* = C_{OA}$ is half in the particles. The model does not impose this equilibrium. It computes
condensation and evaporation kinetically for every size bin, with Raoult's law and the Kelvin
effect, and so relaxes towards it.

Move the time slider in the volatility plot. As C_OA grows (the dashed line moves right), more of
each class moves into the particles.

In [ ]:
plots.volatility(base)

**Check the model against the equation.** For every class that holds mass, the cell compares the
model's particle fraction with $1/(1 + c^*/C_{OA})$, using the model's own C_OA. The run is at
298.15 K, where c\* is 2–3 % above its 298 K value; the cell uses the 298 K values, which accounts
for part of the small gap below the curve.

In [ ]:
time_index = -1   # Try this: 1 (10 min), 6 (1 h) or 18 (3 h); outputs are every 10 min

vol = base.volatility
log10_cstar = np.asarray(vol["log10_cstar"])
particle = np.asarray(vol["particle_ugm3"])[time_index]
gas = np.asarray(vol["gas_ugm3"])[time_index]
c_nv = at.series(base, "POA_ugm3")[time_index]
c_oa = at.series(base, "SOA_ugm3")[time_index] + c_nv

has_mass = particle + gas > 1e-3                 # skip empty classes
fraction = particle[has_mass] / (particle + gas)[has_mass]
x_line = np.linspace(-3.5, 4.5, 200)
fig = go.Figure([
    go.Scatter(x=x_line, y=1 / (1 + 10**x_line / c_oa), name=f"equilibrium, C_OA = {c_oa:.2f} µg m⁻³",
               line=dict(color="#444444", dash="dash")),
    go.Scatter(x=log10_cstar[has_mass], y=fraction, mode="markers",
               name="model", marker=dict(size=10, color=config.PARTICLE_COLOR)),
])
fig.update_xaxes(title="log₁₀ c* at 298 K (µg m⁻³)")
fig.update_yaxes(title="Particle fraction F")
plots.style(fig, f"Particle fraction per c* class at t = {base.time_h[time_index]:g} h")

The model points sit on or just below the equilibrium curve. Vapours are still being made, so the
particles lag a little behind; the 298 K c\* values used here add a little to the gap.

**How sensitive is the SOA to c\*?** If we keep the total (gas + particle) mass of each class from
the run and shift every c\* by the same number of decades, the equilibrium equation gives a new
SOA. The cell solves $C_{OA} = C_{nv} + \sum_j C_{tot,j} / (1 + c^*_j/C_{OA})$ by fixed-point
iteration. It needs no model run, but it ignores how the chemistry would respond.

In [ ]:
def equilibrium_soa(total_ugm3, log10_cstar, c_nv, shift_decades=0.0):
    """SOA [ug/m3] at absorptive equilibrium for class totals `total_ugm3` and c* shifted by `shift_decades`."""
    cstar = 10.0 ** (log10_cstar + shift_decades)
    c_oa = c_nv + total_ugm3.sum()                 # start from "everything condenses"
    for _ in range(1000):
        new = c_nv + np.sum(total_ugm3 / (1 + cstar / c_oa))
        if abs(new - c_oa) < 1e-9 * c_oa:
            break
        c_oa = new
    return c_oa - c_nv


total = particle + gas
print(f"model SOA at t = {base.time_h[time_index]:g} h: {c_oa - c_nv:.2f} µg/m³")
for shift in (-1, 0, 1, 2):   # Try this: other shifts, e.g. 0.5 or -2
    print(f"all c* × 10^{shift:+d}:  equilibrium SOA = {equilibrium_soa(total, log10_cstar, c_nv, shift):6.2f} µg/m³")

Shifting every c\* up by one decade (easily within the uncertainty of the formula) removes about
two thirds of the SOA, and two decades remove almost all of it.

### Where does the SOA go?

The number size distribution shows the seed growing as SOA condenses. The composition plot shows
the dry mass by component. Sulfate grows only a little (H₂SO₄ from SO₂ + OH), and there is no
nitrate because N₂O₅ uptake is off in this level.

In [ ]:
plots.size_distribution(base)   # Try this: quantity="mass"; or plots.banana(base)

In [ ]:
plots.composition(base)   # Try this: fraction=True

## 5. Top SOA contributors

The model keeps track of how much of each product is in the particles, so it can list the species
that make up the SOA. The bars are coloured by whether the formula contains N. The title gives the
N-containing share of the mass shown.

In [ ]:
plots.top_contributors(base)   # Try this: n=5

Nearly all of the SOA here is PANs and organic nitrates. That comes from the placeholder c\*, not
from limonene chemistry:

1. The Li et al. formula lowers c\* sharply for each N atom (the $n_N b_N$ term, plus the O atoms
   that come with –ONO₂ and –C(O)OONO₂). The main non-nitrogen products, such as limonaldehyde
   (10^5.0 µg m⁻³) and limononic acid (10^4.3), come out above the most volatile class
   (10^4.25 µg m⁻³) and are never allowed to condense. Most hydroperoxides come out at
   10^2.5 µg m⁻³ or higher (many above the top class) and stay mostly in the gas.
2. PANs are thermally labile: in the gas phase C822PAN falls apart in about 40 minutes (Section 2).
   Once a PAN is in the particles, the model only lets it evaporate. It does not react or
   decompose there.
3. Other estimation methods, for example group-contribution methods that do see functional groups,
   can differ from Li et al. by one or more decades for the same molecule.

So c\* is the most uncertain input in Level 2, and the next experiment shows how much it matters.

### Under the hood: change c\* yourself

`at.run(..., cstar_ugm3={species: c*})` overrides the Li et al. value species by species (c\* in
µg m⁻³ at 298 K). Behind the scenes it writes a c\* table, a CSV with columns `species,cstar_ugm3`,
and passes it to the model; the curated table will come in the same way. A result made like this
can be rerun with `at.run(result)` like any other.

**Experiment: what if PANs did not condense?** Give every PAN species c\* = 10⁵ µg m⁻³. That is
above the top class, so they stay in the gas phase. The printed *c\* source* then counts the PANs
as "not condensable", which is exactly what this table does.

In [ ]:
pans = [s for s in classes.species if "PAN" in s]
print(f"{len(pans)} condensable PAN species, e.g. {', '.join(pans[:6])}")

new_cstar = {s: 1e5 for s in pans}
# Try this instead: all N-containing products two decades more volatile
# new_cstar = {s: 10 ** (log_cstar[s] + 2) for s in classes.species if has_n[s]}
# or make limononic acid low-volatility: new_cstar = {"LIMONONIC": 0.1}

no_pan = at.run(PRESET, cstar_ugm3=new_cstar, name="PANs stay in the gas")
print("c* source:", no_pan.diagnostics["cstar_source"])
plots.time_series([base, no_pan], ["SOA_ugm3", "dry_ugm3"])

In [ ]:
plots.top_contributors(no_pan)

<details><summary><b>What you should see</b> (open after running)</summary>

SOA drops from about 16 to under 1 µg m⁻³. The SOA that is left is still mostly organic nitrates
(–ONO₂), because those are the next least volatile products under the Li et al. estimate. One
choice about one group of molecules changes the answer more than tenfold. That is why yields and
the NOx dependence in Level 2 have to wait for the curated table.

</details>

## 6. Experiments: change one input, compare runs

Each run takes about 8–15 s. `at.run(PRESET, gases_ppb={...})` changes only the gases you list;
the others keep their preset values. Use `name=` to label a run in the plots.

### 6a. NOx: 0, 1 and 10 ppb

The preset has 2.5 ppb NOx (0.5 NO + 2 NO₂). Here NOx is split 20% NO and 80% NO₂.

In [ ]:
NOX_PPB = [0, 1, 10]   # Try this: add 2.5 (the preset) or 30
NO_SHARE = 0.2         # NO / NOx at the start

nox_runs = [at.run(PRESET, name=f"NOx = {x:g} ppb",
                   gases_ppb={"NO": NO_SHARE * x, "NO2": (1 - NO_SHARE) * x})
            for x in NOX_PPB]
for r in nox_runs:
    print(f"{r.scenario['name']:>15}: SOA after 6 h = {r.totals['SOA_ugm3'][-1]:6.2f} µg/m³, "
          f"N-containing share of top species = {100 * at.nitrogen_share(r):3.0f}%")
plots.time_series(nox_runs, ["SOA_ugm3", "dry_ugm3"])

In [ ]:
plots.time_series(nox_runs, ["LIMONENE", "OH", "NO2", "O3"], log_y=True)

<details><summary><b>What you should see</b></summary>

- **NOx = 0:** limonene is still oxidized (mostly by O₃), but the SOA is only about 0.01 µg m⁻³.
  With the placeholder c\* the non-nitrogen products are too volatile to condense. Chamber
  experiments report substantial SOA from limonene ozonolysis without NOx, so this is an artefact
  of the placeholder c\*, not limonene chemistry.
- **NOx = 10 ppb:** SOA climbs to about 60 µg m⁻³. That is more than the mass of the limonene
  itself (10 ppb ≈ 56 µg m⁻³). This is possible on paper, because the products carry extra O and N
  atoms (about 230–250 g mol⁻¹ against 136 for limonene), but it is a red flag about c\*. More NOx
  also means more OH here, so limonene goes faster.
- The N-containing share stays near 100% whenever there is NOx.

</details>

### 6b. Ozone

O₃ removes most of the limonene (Section 2). What happens with half or double the O₃?

In [ ]:
O3_PPB = [20, 80]   # Try this: 10 or 160 ppb, or hold O3 constant with hold_constant=["O3"]

o3_runs = [base] + [at.run(PRESET, name=f"O₃ = {x:g} ppb", gases_ppb={"O3": x}) for x in O3_PPB]
labels = ["O₃ = 40 ppb (preset)"] + [f"O₃ = {x:g} ppb" for x in O3_PPB]
for r, label in zip(o3_runs, labels):
    print(f"{label:>22}: limonene left after 3 h = {at.value_at(r, 'LIMONENE', 3.0):.3f} ppb, "
          f"SOA after 6 h = {r.totals['SOA_ugm3'][-1]:5.2f} µg/m³")
plots.time_series(o3_runs, ["LIMONENE", "OH", "SOA_ugm3"], labels=labels, log_y=True)

<details><summary><b>What you should see</b></summary>

More O₃ removes limonene faster, and it also raises OH (ozonolysis releases OH, and O₃ photolysis
makes more). The SOA changes much less (roughly 15 → 17 µg m⁻³ from 20 to 80 ppb O₃), because
NOx, not the oxidant, limits the nitrogen-containing products that make up the SOA.

</details>

### 6c. Initial limonene

The mass yield is $Y = \Delta\mathrm{SOA} / \Delta\mathrm{LIM}$, both in µg m⁻³. To convert
ppb to µg m⁻³: $C = \chi\,[M]\,\mathrm{MW}/N_A$ (with unit factors).

In [ ]:
LIMONENE_PPB = [5, 20]   # Try this: 2 or 40 ppb

lim_runs = [at.run(PRESET, name=f"limonene = {x:g} ppb", gases_ppb={"LIMONENE": x}) for x in LIMONENE_PPB]
lim_runs = sorted([base] + lim_runs, key=lambda r: r.scenario["gases_ppb"]["LIMONENE"])

ugm3_per_ppb = at.ugm3_per_ppb(mech.molecular_weight("LIMONENE"), base)   # molar mass in g/mol
print(f"1 ppb limonene = {ugm3_per_ppb:.2f} µg/m³ at 298 K, 1 atm\n")
print("limonene  reacted   SOA    yield")
for r in lim_runs:
    lim = at.series(r, "LIMONENE")
    reacted = (lim[0] - lim[-1]) * ugm3_per_ppb
    soa = r.totals["SOA_ugm3"][-1]
    print(f"{lim[0]:5.0f} ppb  {reacted:5.1f}  {soa:6.2f}  {soa / reacted:6.2f}   (µg/m³)")
plots.time_series(lim_runs, ["SOA_ugm3", "OH"])

<details><summary><b>What you should see</b></summary>

Limonene quadruples from 5 to 20 ppb, but the SOA barely changes (about 15 → 16 µg m⁻³), so the
yield falls from about 0.5 to about 0.15. Do **not** read this as the classic dependence of yield
on organic loading (Odum et al., 1996). The SOA is capped because nearly all of it contains
nitrogen and the 2.5 ppb of NOx runs out. Once there is a curated c\* table, this experiment
should be repeated.

</details>

### 6d. Seed loading

Change the number of seed particles (same size and composition). The seed carries 30% non-volatile
organic, so a bigger seed also brings more absorbing mass at the start.

In [ ]:
SEED_NUMBER_CM3 = [500, 10000]   # Try this: 100 or 30000; or add "gmd_nm" to the seed dict

seed_runs = [at.run(PRESET, name=f"seed {n:g} cm⁻³", seed={"number_cm3": n}) for n in SEED_NUMBER_CM3]
seed_runs = sorted([base] + seed_runs, key=lambda r: r.scenario["seed"]["modes"][0]["number_cm3"])
for r in seed_runs:
    s = at.summary(r)
    print(f"seed {r.scenario['seed']['modes'][0]['number_cm3']:6.0f} cm⁻³: POA = {s['POA_ugm3_start']:.2f}, "
          f"SOA after 6 h = {s['SOA_ugm3_end']:5.2f} µg/m³, GMD {s['gmd_nm_start']:.0f} → {s['gmd_nm_end']:.0f} nm")
plots.compare_size_distributions(seed_runs)

<details><summary><b>What you should see</b></summary>

The SOA hardly changes (about 15.6 to 16.6 µg m⁻³) while the number of particles changes 20-fold.
What changes is how the SOA is shared out. A few particles grow a lot (to about 290 nm with
500 cm⁻³), and many particles grow a little (to about 110 nm with 10 000 cm⁻³). Each run starts
from its own seed (dotted lines). The seed's organic (0.2–3.8 µg m⁻³) is small next to the SOA
itself (about 16 µg m⁻³), and most particle SOA sits in classes with c\* of 3 µg m⁻³ or less, which
are more than 80% condensed at this C_OA. Do not generalize this: when the organic loading is low
or the products are more volatile, the absorbing mass matters a lot. In Level 1, try c\* = 10 with
a 2000 cm⁻³, 150 nm seed: the SOA rises about tenfold (0.2 → 2.5 µg m⁻³).

</details>

## 7. Run it yourself

Pick limonene, NOx and O₃ and press **Run model** (about 10–20 s). The plots compare your last three
runs with the preset. Every run is kept in the list `widget_runs`, so you can use it with any plot
function afterwards, e.g. `plots.top_contributors(widget_runs[-1])`.

In [ ]:
import html

import ipywidgets as widgets
from IPython.display import clear_output, display

WIDGET_NO_SHARE = 0.2   # NO / NOx at the start, as in Section 6a
limonene_in = widgets.SelectionSlider(options=[1, 2, 5, 10, 20, 40], value=10, description="Limonene (ppb)")
nox_in = widgets.SelectionSlider(options=[0, 0.5, 1, 2.5, 5, 10, 20], value=2.5, description="NOx (ppb)")
o3_in = widgets.SelectionSlider(options=[10, 20, 40, 80, 160], value=40, description="O₃ (ppb)")
for slider in (limonene_in, nox_in, o3_in):
    slider.style.description_width = "110px"
    slider.layout.width = "420px"
run_button = widgets.Button(description="Run model", button_style="primary", icon="play")
status = widgets.HTML("Pick values and press <b>Run model</b>.")
output = widgets.Output()
widget_runs = []


def on_run(_):
    run_button.disabled = True
    status.value = "Running… (about 10–20 s)"
    lim, nox, o3 = limonene_in.value, nox_in.value, o3_in.value
    try:
        result = at.run(PRESET, progress=False, name=f"lim {lim:g}, NOx {nox:g}, O₃ {o3:g} ppb",
                        gases_ppb={"LIMONENE": lim, "O3": o3,
                                   "NO": WIDGET_NO_SHARE * nox, "NO2": (1 - WIDGET_NO_SHARE) * nox})
    except Exception as err:   # show the error here; widget callbacks otherwise hide it
        status.value = f"<b>Run failed:</b> {html.escape(str(err))}"
        return
    finally:
        run_button.disabled = False
    widget_runs.append(result)
    status.value = (f"Done in {result.diagnostics['runtime_s']:.0f} s. SOA after 6 h: "
                    f"<b>{result.totals['SOA_ugm3'][-1]:.2f} µg m⁻³</b>")
    with output:
        clear_output(wait=True)
        recent = [base] + widget_runs[-3:]
        plots.time_series(recent, ["SOA_ugm3", "dry_ugm3"]).show()
        plots.time_series(recent, ["LIMONENE", "OH", "NO2"], log_y=True).show()
        plots.top_contributors(result).show()


run_button.on_click(on_run)
display(widgets.VBox([limonene_in, nox_in, o3_in, widgets.HBox([run_button, status]), output]))

## Questions to think about

1. O₃ removes most of the limonene, yet the SOA depends mostly on NOx. How can both be true in this
   model?
2. Which properties of a product decide its particle fraction? Which of them does the Li et al.
   formula know about?
3. PANs decompose in the gas phase within about 40 minutes. What could happen to a PAN in a real
   particle, and what does the model assume?
4. Suppose a curated table raised the c\* of the PANs and lowered the c\* of the acids and
   hydroperoxides. How would you expect the NOx experiment (6a) and the limonene experiment (6c) to
   change?
5. The seed experiment changed particle number 20-fold but SOA by only a few percent. When would
   the seed's organic mass matter more?
6. Light is constant here. Which results would change under real sunlight with a night (Level 3)?

## Going further

**The raw model API.** `at.run` is a thin wrapper. Underneath, a run is a `Scenario` (frozen
dataclasses, loaded from a TOML preset) passed to `tomas_jax.box.driver.run`. One difference: the
raw `Scenario.replace(gases_ppb=...)` replaces the *whole* gas dictionary, while `at.run` merges it
into the preset's gases.

In [ ]:
from tomas_jax.box import driver, scenario

s = scenario.load_preset(PRESET).replace(duration_h=1.0, photolysis={"jno2_per_s": 8e-3})
print(s.photolysis)
raw = driver.run(s)   # what at.run calls, without the progress bar and timeout
print(f"{raw.diagnostics['runtime_s']:.1f} s; SOA after 1 h = {raw.totals['SOA_ugm3'][-1]:.2f} µg/m³")

**More things to try**

- Dimmer or brighter light: `at.run(PRESET, photolysis={"jno2_per_s": 2e-3})` (clear-sky midday
  summer sun gives about 1×10⁻² s⁻¹ in TUV-x; see Level 3).
- Hold a gas at its initial value: `at.run(PRESET, hold_constant=["O3"])` or `["NO2"]` (a
  stand-in for continuous emissions).
- Temperature: `environment={"temp_k": 288}`. Every c\* changes with T by Clausius–Clapeyron
  (ΔH_vap = 131 − 11 log₁₀ c\* kJ mol⁻¹).
- Seed composition: `seed={"organic_mass_fraction": 0.9}`.
- Save a run and load it later: `at.save_result(r, "my_run.json")`, `at.load_result("my_run.json")`.
- See exactly what a run will use without running it: `at.describe(at.scenario_from(PRESET, gases_ppb={"NO": 0}))`.
- Read more: [ASSUMPTIONS.md](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/ASSUMPTIONS.md) (A1–A14),
  [gaschem.md](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/docs/aaar/gaschem.md) (mechanism, solver, photolysis, volatility) and
  [soa_partitioning.md](https://github.com/reflective-org/TOMAS-JAX/blob/87537364420e76cd50203049e4d74cb23660327d/docs/aaar/soa_partitioning.md) (the kinetic partitioning).
- Back to partitioning with a single surrogate: [Level 1 · Volatility](01_volatility.ipynb).
- Next: [Level 3 · Day & Night](03_day_and_night.ipynb) adds the real diurnal cycle of sunlight,
  night-time NO₃ chemistry and N₂O₅ uptake. [Inside the model](04_inside_the_model.ipynb) goes
  deeper into the code.

**References**

- Jenkin, M. E., Saunders, S. M., and Pilling, M. J. (1997), Atmos. Environ. 31, 81–104; Saunders,
  S. M., et al. (2003), ACP 3, 161–180: the Master Chemical Mechanism (https://mcm.york.ac.uk).
- Li, Y., Pöschl, U., and Shiraiwa, M. (2016), Molecular corridors and parameterizations of
  volatility in the chemical evolution of organic aerosols, ACP 16, 3327–3344.
- Pankow, J. F. (1994), Atmos. Environ. 28, 185–188; Donahue, N. M., et al. (2006), ES&T 40,
  2635–2643: absorptive partitioning and the volatility basis set.
- Odum, J. R., et al. (1996), ES&T 30, 2580–2585: SOA yields and organic mass.
- Epstein, S. A., Riipinen, I., and Donahue, N. M. (2010), ES&T 44, 743–748: ΔH_vap(c\*).